# PrecisionSkin - Localization Dataset Audit & Safe Reorganisation

**What this notebook does:** inspects every dataset you downloaded for Acne / Eczema / Hyperpigmentation, shows you the real
images with their boxes or masks, audits labels and annotations, finds duplicates, and *only then* builds a clean,
standardised copy with full provenance.

**What it never does**
- It never modifies, moves or deletes your original folders. Originals are opened **read-only**; every output goes under `OUT_ROOT`.
- It does not train anything and does **not** create train/validation/test splits. It stores a `duplicate_group` so that, when you
  split later, copies of the same photo can never land in different splits.
- It never silently renames labels, never deletes suspicious images, and never guesses an anatomical site.

**How to use it:** run the cells top to bottom, one at a time. Each stage prints what it found. Nothing is copied until you set
`APPLY_CHANGES = True` in the config cell *and* run the "apply" stage. Stages 1-12 only read your data and write small CSV reports.

| Stage | What it does |
|---|---|
| 0 | Config |
| 1 | Scan: recursive folder tree, formats, class names, existing splits |
| 2 | Archives (safe extraction to a raw folder; dry-run unless you allow it) |
| 3 | Parse annotations (YOLO boxes/polygons, COCO, Pascal VOC) |
| 4 | Label audit + **proposed** mapping (you confirm) |
| 5 | Per-image features: hashes, size, corruption, face check |
| 6 | Annotation + image validation -> issue list |
| 7 | Exact + near-duplicate groups |
| 8 | Facial / non-facial / needs-manual-review |
| 9 | **Visual browser** - go through every image with its annotations |
| 10 | Review CSV (you decide keep / reject / review) |
| 11 | Proposed final folder structure |
| 12 | Dry-run -> apply (copy only) |
| 13 | Final audit report |

## Stage 0 - Config  (edit this cell only)

In [ ]:
from pathlib import Path
import os

# --- where your downloaded datasets are (READ ONLY - nothing here is ever changed) -------------------------------
SOURCES_ROOT = Path("/mnt/c/Users/Bautista/Downloads/Localization Training Datasets")

# --- where ALL outputs go (must be OUTSIDE SOURCES_ROOT). On WSL a Linux path is much faster than /mnt/c. ---------
#     View it from Windows at:  \\wsl$\Ubuntu\home\<you>\PrecisionSkinV1\localization_dataset
OUT_ROOT = Path.home() / "PrecisionSkinV1" / "localization_dataset"

# --- SAFETY SWITCH: while False, nothing is copied or extracted. Reports (small CSV files) are always written. ---
APPLY_CHANGES = False

# --- quick trial: set e.g. 300 to audit only 300 random images per source first, then None for everything ---------
LIMIT_IMAGES_PER_SOURCE = None
RUN_FACE_DETECTION = True            # slowest step (a few minutes for ~17k images); results are cached
# OpenCV 5 no longer ships the Haar detector, so the face check uses the YuNet model that is already in your app folder:
FACE_MODEL_PATH = "/mnt/c/Users/Bautista/source/repos/SkinDevApp/SkinDevApp/face_detection_yunet_2023mar.onnx"
NEAR_DUP_MIN_CORR = 0.97             # thumbnail correlation (best of 8 rotations/flips). Calibrated on your data: 0.97 = same photo,
                                     # <=0.93 starts chaining different photos together. Do not lower it without looking at the groups.
MAX_GROUP_SIZE = 150                 # safety net: a union that would create a bigger group is skipped (and reported)
USE_NAME_GROUPS = True               # also group Roboflow copies by original file name (e.g. 'acne-269_jpeg')
BROWSER_COLUMNS = 4
ALLOW_UNCONFIRMED_LABELS = False     # Stage 12 refuses to copy while any label mapping is still unconfirmed (True only for testing)

# --- which condition does each SOURCE FOLDER contain? Guessed from the folder name; override if wrong -------------
CONDITION_HINTS = {
    "Acne":              ["acne", "pimple"],
    "Eczema":            ["eczema", "dermatitis"],
    "Hyperpigmentation": ["hyperpig", "pigment", "melasma", "dark spot"],
}
CONDITION_OVERRIDE = {
    # "some folder name": "Eczema",
}
FACIAL_DECLARED_NAME_WORDS = ["facial"]   # a source whose folder name says 'facial' is recorded as facial (basis: source name)

# --- label mapping: leave empty first. After Stage 4 shows the proposal, put your decisions here. -----------------
#     key = (source folder name, original label)   value = "Acne" | "Eczema" | "Hyperpigmentation" | "DROP"
LABEL_MAP_OVERRIDES = {
    # Decided after looking at sample images of every label (change any line if you disagree):
    ("ACNE Detection.v2i.yolov8", "0"): "Acne",                                   # pimples/blemishes (many grayscale/noisy augmented copies)
    ("acne-detection.v2i.yolov8", "0"): "Acne",                                   # small pimple-like spots on faces
    ("acne-detection.v2i.yolov8", "1"): "Acne",
    ("acne-detection.v2i.yolov8", "2"): "Acne",
    ("acne-detection.v2i.yolov8", "spot"): "Acne",                                # least certain: skim a few with BROWSE_FILTER["label"]
    ("Hyperpigmentation.v1-hyperpigmentation.yolov8", "3"): "Hyperpigmentation",  # real melasma/sun-spot patches, NOT junk
    ("Pigmentation.v1i.yolov8", "0"): "Hyperpigmentation",                        # brown cheek patches (polygons)
    ("hyperpigmentation_yolov10.v3-hyperpigmentation_yolov10_base.yolov8", "redness"): "DROP",   # only 1 box in the whole dataset
}

# --- anatomical site, per SOURCE FOLDER: use ONLY after you have browsed that source and confirmed it is facial. ------
#     e.g. {"Pigmentation.v1i.yolov8": "facial"}   (recorded as 'declared by you for this source'; images flagged by other checks still get reviewed)
SOURCE_SITE_DECLARATION = {
    # Decided from ~16 random images of each source that the face detector could not confirm (close-ups of cheeks, nose, chin, jaw, forehead).
    # Sample-based: for the two acne sources and Pigmentation.v1i roughly 1 image in 6 looked unclear (could be neck/back). The facial-only
    # TEST sets are hand-boxed and verified separately, so this does not affect the final evaluation.
    "hyperpigmentation_yolov10.v3-hyperpigmentation_yolov10_base.yolov8": "facial",
    "hyperpigmentation.v2i.yolov8": "facial",
    "Hyperpigmentation.v1-hyperpigmentation.yolov8": "facial",
    "Pigmentation.v1i.yolov8": "facial",
    "ACNE Detection.v2i.yolov8": "facial",
    "acne-detection.v2i.yolov8": "facial",
}

# --- anatomical site, per IMAGE: image_id -> "facial" | "non_facial" | "needs_manual_review". Filled by YOU after browsing. ---
SITE_OVERRIDES = {
    # "IMG-001234": "non_facial",
}

# suspicious-box thresholds (fraction of the image area)
LARGE_BOX_FRAC = {"Acne": 0.10, "Eczema": 0.85, "Hyperpigmentation": 0.85}
LOW_RES_PX = 128

STANDARD_CONDITIONS = ["Acne", "Eczema", "Hyperpigmentation"]

RAW_DIR = OUT_ROOT / "00_raw_extracted"
AUDIT_DIR = OUT_ROOT / "01_audit_reports"
REVIEW_DIR = OUT_ROOT / "02_review_queue"
FINAL_DIR = OUT_ROOT / "03_standardized"

# ---- hard safety checks --------------------------------------------------------------------------------------
SOURCES_ROOT = SOURCES_ROOT.resolve()
OUT_ROOT = OUT_ROOT.resolve()
assert SOURCES_ROOT.exists(), "SOURCES_ROOT not found: %s" % SOURCES_ROOT
assert OUT_ROOT != SOURCES_ROOT and SOURCES_ROOT not in OUT_ROOT.parents and OUT_ROOT not in SOURCES_ROOT.parents, \
    "OUT_ROOT must be a separate folder, not inside (or containing) SOURCES_ROOT"

def safe_out(p):
    """Every write goes through this: refuses any path outside OUT_ROOT."""
    p = Path(p).resolve()
    assert OUT_ROOT in p.parents or p == OUT_ROOT, "refusing to write outside OUT_ROOT: %s" % p
    return p

print("SOURCES_ROOT :", SOURCES_ROOT, "(read-only)")
print("OUT_ROOT     :", OUT_ROOT)
print("APPLY_CHANGES:", APPLY_CHANGES)

In [ ]:
import collections, json, re, hashlib, shutil, zipfile, math, pickle, random, threading, time
import xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
try:
    import yaml
except ImportError:
    yaml = None
try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 70)

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
ARCH_EXT = {".zip", ".rar", ".7z", ".tar", ".gz", ".tgz"}
SPLIT_NAMES = {"train", "valid", "val", "validation", "test"}

def imread_any(p):
    data = np.fromfile(str(p), np.uint8)
    return cv2.imdecode(data, cv2.IMREAD_COLOR)

def infer_condition(name):
    if name in CONDITION_OVERRIDE:
        return CONDITION_OVERRIDE[name]
    n = name.lower()
    hits = [c for c, ws in CONDITION_HINTS.items() if any(w in n for w in ws)]
    return hits[0] if len(hits) == 1 else None

## Stage 1 - Scan (read-only)
Recursive scan of every source folder: sub-folders, image counts, annotation files by type, archives, `data.yaml`/class names and any
existing train/valid/test folders. Nothing is modified.

In [ ]:
def count_by_ext(files):
    c = collections.Counter(Path(f).suffix.lower() for f in files)
    return dict(sorted(c.items(), key=lambda kv: -kv[1]))

def read_yaml_names(p):
    txt = Path(p).read_text(encoding="utf-8", errors="ignore")
    names = None
    if yaml is not None:
        try:
            names = (yaml.safe_load(txt) or {}).get("names")
        except Exception:
            names = None
    if names is None:
        m = re.search(r"^names:\s*(\[.*\])", txt, re.M)
        if m:
            names = [x.strip(" '\"") for x in m.group(1).strip("[]").split(",")]
    if isinstance(names, list):
        return {i: str(n) for i, n in enumerate(names)}
    if isinstance(names, dict):
        return {int(k): str(v) for k, v in names.items()}
    return {}

def walk_files(root):
    """One fast directory walk (os.walk uses scandir; far faster than rglob+is_file on WSL's /mnt/c)."""
    out = []
    for dp, _dn, fn in os.walk(root):
        dp = Path(dp)
        out.extend(dp / f for f in fn)
    return out

SOURCE_DIRS = {}          # source name -> folder
FILE_INDEX = {}           # source name -> every file (walked ONCE, reused by all later stages)
SCAN = []
archives_found = []

for entry in sorted(SOURCES_ROOT.iterdir()):
    if entry.is_dir():
        SOURCE_DIRS[entry.name] = entry
    elif entry.suffix.lower() in ARCH_EXT:
        archives_found.append(entry)

print("=== %d source folder(s) in %s ===\n" % (len(SOURCE_DIRS), SOURCES_ROOT))
for name, root in SOURCE_DIRS.items():
    files = walk_files(root)
    FILE_INDEX[name] = files
    exts = count_by_ext(files)
    imgs = [p for p in files if p.suffix.lower() in IMG_EXT]
    yamls = [p for p in files if p.suffix.lower() in (".yaml", ".yml")]
    jsons = [p for p in files if p.suffix.lower() == ".json"]
    xmls = [p for p in files if p.suffix.lower() == ".xml"]
    txts = [p for p in files if p.suffix.lower() == ".txt"]
    arch = [p for p in files if p.suffix.lower() in ARCH_EXT]
    splits = sorted({part for p in imgs for part in p.relative_to(root).parts[:-1] if part.lower() in SPLIT_NAMES})
    names = read_yaml_names(root / "data.yaml") if (root / "data.yaml").exists() else {}
    fmt = []
    if (root / "data.yaml").exists() or (yamls and txts):
        fmt.append("YOLO txt (boxes and/or polygons)")
    if jsons:
        fmt.append("JSON (COCO?)")
    if xmls:
        fmt.append("XML (Pascal VOC?)")
    mask_dirs = sorted({p.parent.name for p in imgs if "mask" in p.parent.name.lower()})
    if mask_dirs:
        fmt.append("mask images in %s (NOT parsed by this notebook)" % mask_dirs)
    if not fmt:
        fmt.append("no annotation files found (images only)")
    SCAN.append({"source": name, "condition_guess": infer_condition(name) or "UNKNOWN - set CONDITION_OVERRIDE",
                 "images": len(imgs), "annotation_files": len(txts) + len(jsons) + len(xmls), "format": "; ".join(fmt),
                 "splits_present": ", ".join(splits) or "none", "class_names": names or "(none found)",
                 "archives_inside": len(arch), "extra_yaml": [p.name for p in yamls if p.name != "data.yaml"]})
    print("%s\n  condition guess : %s\n  images          : %d   file types: %s" % (name, SCAN[-1]["condition_guess"], len(imgs), exts))
    print("  annotation files: %d txt, %d json, %d xml | format: %s" % (len(txts), len(jsons), len(xmls), SCAN[-1]["format"]))
    print("  existing splits : %s | class names: %s" % (SCAN[-1]["splits_present"], SCAN[-1]["class_names"]))
    if SCAN[-1]["extra_yaml"]:
        print("  extra yaml files: %s  (only data.yaml is used for class names)" % SCAN[-1]["extra_yaml"])
    # folder tree (counts per folder, depth <= 3)
    per_dir = collections.defaultdict(lambda: [0, 0])
    for p in files:
        per_dir[p.parent][0 if p.suffix.lower() in IMG_EXT else 1] += 1
    for d in sorted(per_dir):
        rel = d.relative_to(root)
        if len(rel.parts) <= 3:
            print("    %-40s %6d images, %6d annotation/other files" % (str(rel) if str(rel) != "." else ".", per_dir[d][0], per_dir[d][1]))
    print()

if archives_found:
    print("Archives at the top level:", [a.name for a in archives_found])
df_scan = pd.DataFrame(SCAN)
display(df_scan[["source", "condition_guess", "images", "annotation_files", "splits_present", "format"]])

## Stage 2 - Archives
ZIP files are extracted into `00_raw_extracted/` (never next to your originals). RAR/7z are listed but not extracted here (extract them
yourself into a new folder inside `SOURCES_ROOT`, or tell me). While `APPLY_CHANGES = False` this is a **dry run**.

In [ ]:
all_archives = list(archives_found)
for name in list(SOURCE_DIRS):
    all_archives += [p for p in FILE_INDEX[name] if p.suffix.lower() in ARCH_EXT]

if not all_archives:
    print("No archives found. Nothing to extract.")
for a in all_archives:
    target = RAW_DIR / a.stem
    if a.suffix.lower() != ".zip":
        print("SKIP (cannot extract %s here): %s" % (a.suffix, a))
        continue
    with zipfile.ZipFile(a) as z:
        bad = [m for m in z.namelist() if m.startswith("/") or ".." in Path(m).parts]
        n = len(z.namelist())
    if bad:
        print("REFUSED (unsafe paths inside zip):", a); continue
    if APPLY_CHANGES:
        safe_out(target).mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(a) as z:
            z.extractall(target)
        SOURCE_DIRS[a.stem] = target
        FILE_INDEX[a.stem] = walk_files(target)
        print("extracted %d files -> %s" % (n, target))
    else:
        print("DRY-RUN: would extract %d files from %s -> %s  (set APPLY_CHANGES=True, rerun Stage 1-2)" % (n, a.name, target))

## Stage 3 - Parse annotations
Reads YOLO (`.txt`, boxes **and** polygons), COCO (`.json`, boxes and polygons) and Pascal VOC (`.xml`) into **one** table of images and one
table of annotations. Original labels are kept exactly as found.

In [ ]:
def parse_yolo_file(lp, names):
    anns, probs = [], []
    for i, line in enumerate(Path(lp).read_text(errors="ignore").splitlines()):
        t = line.split()
        if not t:
            continue
        try:
            vals = [float(x) for x in t]
        except ValueError:
            probs.append(("malformed_annotation", "non-numeric content on line %d" % (i + 1))); continue
        cid, v = int(vals[0]), vals[1:]
        label = names.get(cid, "id_%d" % cid)
        if cid not in names and names:
            probs.append(("malformed_annotation", "class id %d not in data.yaml" % cid))
        if len(v) == 4:
            anns.append({"label_id": cid, "original_label": label, "type": "bbox", "cx": v[0], "cy": v[1], "w": v[2], "h": v[3], "poly": None})
        elif len(v) >= 6 and len(v) % 2 == 0:
            xs, ys = v[0::2], v[1::2]
            anns.append({"label_id": cid, "original_label": label, "type": "polygon",
                         "cx": (min(xs) + max(xs)) / 2, "cy": (min(ys) + max(ys)) / 2,
                         "w": max(xs) - min(xs), "h": max(ys) - min(ys), "poly": [[x, y] for x, y in zip(xs, ys)]})
        else:
            probs.append(("malformed_annotation", "line %d has %d numbers" % (i + 1, len(v))))
    return anns, probs

def load_coco(jp):
    """-> {absolute image path: dict(width,height,anns,probs)}  (COCO image folder = the json's own folder)"""
    j = json.loads(Path(jp).read_text(errors="ignore"))
    if not (isinstance(j, dict) and "images" in j and "annotations" in j):
        return None
    cats = {c["id"]: c["name"] for c in j.get("categories", [])}
    per = collections.defaultdict(list)
    for a in j["annotations"]:
        per[a["image_id"]].append(a)
    out = {}
    for im in j["images"]:
        W, H = float(im.get("width") or 0), float(im.get("height") or 0)
        anns, probs = [], []
        for a in per.get(im["id"], []):
            lab = cats.get(a.get("category_id"), "id_%s" % a.get("category_id"))
            if not W or not H:
                probs.append(("malformed_annotation", "image size missing in COCO json")); continue
            x, y, w, h = a.get("bbox", [0, 0, 0, 0])
            seg = a.get("segmentation")
            base = {"label_id": a.get("category_id"), "original_label": lab, "cx": (x + w / 2) / W, "cy": (y + h / 2) / H, "w": w / W, "h": h / H}
            if isinstance(seg, list) and seg and isinstance(seg[0], list):
                for s in seg:
                    xs, ys = s[0::2], s[1::2]
                    anns.append(dict(base, type="polygon", poly=[[px / W, py / H] for px, py in zip(xs, ys)]))
            else:
                if isinstance(seg, dict):
                    probs.append(("unsupported_annotation", "RLE mask in COCO json (bbox kept, mask not parsed)"))
                anns.append(dict(base, type="bbox", poly=None))
        out[os.path.normpath(str(Path(jp).parent / im["file_name"]))] = {"width": int(W), "height": int(H), "anns": anns, "probs": probs}
    return out

def parse_voc(xp):
    r = ET.parse(xp).getroot()
    sz = r.find("size")
    W, H = float(sz.find("width").text), float(sz.find("height").text)
    anns = []
    for o in r.findall("object"):
        bb = o.find("bndbox")
        x0, y0, x1, y1 = [float(bb.find(k).text) for k in ("xmin", "ymin", "xmax", "ymax")]
        anns.append({"label_id": None, "original_label": o.find("name").text, "type": "bbox",
                     "cx": (x0 + x1) / 2 / W, "cy": (y0 + y1) / 2 / H, "w": (x1 - x0) / W, "h": (y1 - y0) / H, "poly": None})
    return {"width": int(W), "height": int(H), "anns": anns, "probs": []}

img_rows, ann_rows, issue_rows, orphan_rows = [], [], [], []
rng = random.Random(0)

for sname, root in SOURCE_DIRS.items():
    cond = infer_condition(sname)
    names = read_yaml_names(root / "data.yaml") if (root / "data.yaml").exists() else {}
    coco_map = {}
    fl = FILE_INDEX[sname]
    fset = set(str(p) for p in fl)
    for jp in [p for p in fl if p.suffix.lower() == ".json"]:
        try:
            m = load_coco(jp)
        except Exception as e:
            print("  could not read json %s: %s" % (jp.name, e)); m = None
        if m:
            coco_map.update(m)
    files = sorted(p for p in fl if p.suffix.lower() in IMG_EXT)
    if LIMIT_IMAGES_PER_SOURCE and len(files) > LIMIT_IMAGES_PER_SOURCE:
        files = sorted(rng.sample(files, LIMIT_IMAGES_PER_SOURCE))
    used_labels = set()
    for ip in files:
        key = os.path.normpath(str(ip))
        anns, probs, fmt, apath, W, H = [], [], "none", "", None, None
        lp = ip.parent.parent / "labels" / (ip.stem + ".txt") if ip.parent.name.lower() == "images" else ip.with_suffix(".txt")
        xp = ip.with_suffix(".xml")
        if key in coco_map:
            d = coco_map[key]; anns, probs, W, H = d["anns"], list(d["probs"]), d["width"], d["height"]; fmt = "coco"; apath = "(coco json)"
        elif str(lp) in fset:
            anns, probs = parse_yolo_file(lp, names); used_labels.add(str(lp)); apath = str(lp)
            fmt = "yolo-polygon" if any(a["type"] == "polygon" for a in anns) else "yolo-bbox"
            if not anns and not probs:
                probs.append(("empty_label_file", "label file exists but is empty"))
        elif str(xp) in fset:
            try:
                d = parse_voc(xp); anns, probs, W, H = d["anns"], d["probs"], d["width"], d["height"]; fmt = "voc"; apath = str(xp)
            except Exception as e:
                probs.append(("malformed_annotation", "VOC xml unreadable: %s" % e))
        if not anns and not any(p[0] == "empty_label_file" for p in probs):
            probs.append(("no_annotation", "image has no annotation (missing or empty label)"))
        rel = str(ip.relative_to(root))
        split = next((p for p in ip.relative_to(root).parts[:-1] if p.lower() in SPLIT_NAMES), "")
        img_rows.append({"source": sname, "source_condition": cond, "rel_path": rel, "source_path": key, "original_filename": ip.name,
                         "source_split": split, "annotation_path": apath, "annotation_format": fmt, "width": W, "height": H,
                         "_probs": probs, "_anns": anns})
    # annotation files without an image
    if LIMIT_IMAGES_PER_SOURCE is None:
        for lp in [p for p in fl if p.parent.name == "labels" and p.suffix.lower() == ".txt"]:
            if str(lp) not in used_labels:
                orphan_rows.append({"source": sname, "file": str(lp.relative_to(root)), "issue": "annotation_without_image"})
    print("%-70s images %6d | annotated %6d" % (sname, len(files), sum(1 for r in img_rows if r["source"] == sname and r["_anns"])))

df_img = pd.DataFrame(img_rows).sort_values(["source", "rel_path"]).reset_index(drop=True)
df_img.insert(0, "image_id", ["IMG-%06d" % (i + 1) for i in range(len(df_img))])
ann_flat = []
for _, r in df_img.iterrows():
    for k, a in enumerate(r["_anns"]):
        ann_flat.append(dict(image_id=r["image_id"], source=r["source"], ann_idx=k, **a))
df_ann = pd.DataFrame(ann_flat)
print("\nimages: %d | annotations: %d (bbox %d, polygon %d) | orphan annotation files: %d" % (
    len(df_img), len(df_ann), (df_ann.type == "bbox").sum(), (df_ann.type == "polygon").sum(), len(orphan_rows)))

## Stage 4 - Label audit and **proposed** mapping
Shows every original label found per source. Labels that clearly name a condition are mapped by keyword. Numeric or vague labels (`0`, `1`, `spot`,
`redness`...) are **not** silently renamed: they get the folder's condition as a *proposal* flagged `needs_confirmation = YES`. Put your decisions in
`LABEL_MAP_OVERRIDES` (Stage 0), re-run this cell, and every annotation keeps `original_label` + a mapping history.

In [ ]:
KEYWORDS = {
    "Acne": ["acne", "pimple", "comedo", "pustule", "papule", "blackhead", "whitehead", "nodule", "cyst"],
    "Eczema": ["eczema", "dermatitis", "atopic"],
    "Hyperpigmentation": ["hyperpig", "melasma", "pigment", "dark spot", "pih", "chloasma"],
}
def propose(source_cond, label):
    low = str(label).lower()
    hits = [c for c, ws in KEYWORDS.items() if any(w in low for w in ws)]
    if len(hits) == 1:
        conflict = source_cond is not None and hits[0] != source_cond
        return hits[0], "label text names the condition" + ("; CONFLICTS with folder condition %s" % source_cond if conflict else ""), "YES" if conflict else "no"
    if source_cond:
        return source_cond, "vague/numeric label -> folder condition used as a PROPOSAL", "YES"
    return "UNMAPPED", "no keyword and folder condition unknown", "YES"

src_cond = df_img.groupby("source")["source_condition"].first().to_dict()
rows = []
for (s, lab), g in df_ann.groupby(["source", "original_label"]):
    prop, basis, needs = propose(src_cond.get(s), lab)
    ov = LABEL_MAP_OVERRIDES.get((s, lab))
    rows.append({"source": s, "original_label": lab, "annotations": len(g), "images": g.image_id.nunique(),
                 "proposed": prop, "basis": basis, "needs_confirmation": needs,
                 "YOUR_OVERRIDE": ov or "", "EFFECTIVE": ov or prop,
                 "confirmed": "yes" if (ov or needs == "no") else "NO - please decide"})
df_labels = pd.DataFrame(rows)
display(df_labels)
n_open = (df_labels.confirmed != "yes").sum()
print("\n%d label(s) still need your confirmation. They are tagged 'label_needs_confirmation' in the issue list (nothing is dropped)." % n_open)
print("Set LABEL_MAP_OVERRIDES = {(source, label): 'Acne'|'Eczema'|'Hyperpigmentation'|'DROP'} in Stage 0 and re-run this cell.")

eff = {(r.source, r.original_label): r.EFFECTIVE for r in df_labels.itertuples()}
conf = {(r.source, r.original_label): r.confirmed for r in df_labels.itertuples()}
bas = {(r.source, r.original_label): r.basis for r in df_labels.itertuples()}
df_ann["standardized_label"] = [eff[(s, l)] for s, l in zip(df_ann.source, df_ann.original_label)]
df_ann["label_confirmed"] = [conf[(s, l)] == "yes" for s, l in zip(df_ann.source, df_ann.original_label)]
df_ann["mapping_history"] = ["%s: '%s' -> %s [%s]" % (s, l, eff[(s, l)], bas[(s, l)]) for s, l in zip(df_ann.source, df_ann.original_label)]
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
df_labels.to_csv(safe_out(AUDIT_DIR / "label_mapping_proposal.csv"), index=False)
print("saved", AUDIT_DIR / "label_mapping_proposal.csv")

## Stage 5 - Per-image features (hashes, size, corruption, face check)
Reads every image once (read-only) and caches the result in `01_audit_reports/features_cache.pkl`, so re-running is fast. Computes: MD5, width/height,
whether it decodes, a 32x32 thumbnail for duplicate detection, brightness/sharpness, a dermoscopy-vignette heuristic and an
OpenCV face check (tries 4 rotations + profile). **The face check is a heuristic - it is only used to *flag* images, never to delete or relabel.**

In [ ]:
CACHE = AUDIT_DIR / "features_cache.pkl"
cache = pickle.load(open(CACHE, "rb")) if CACHE.exists() else {}
_tl = threading.local()

FACE_BACKEND = None
if RUN_FACE_DETECTION:
    if hasattr(cv2, "FaceDetectorYN") and Path(FACE_MODEL_PATH).exists():
        FACE_BACKEND = "yunet"
    elif hasattr(cv2, "CascadeClassifier"):
        FACE_BACKEND = "haar"
print("face backend:", FACE_BACKEND or "NONE - every image not declared facial will be 'needs_manual_review'")

def _detector():
    if not hasattr(_tl, "d"):
        if FACE_BACKEND == "yunet":
            _tl.d = cv2.FaceDetectorYN.create(str(FACE_MODEL_PATH), "", (320, 320), 0.6, 0.3, 5000)
        else:
            dd = cv2.data.haarcascades
            _tl.d = (cv2.CascadeClassifier(dd + "haarcascade_frontalface_default.xml"), cv2.CascadeClassifier(dd + "haarcascade_profileface.xml"))
    return _tl.d

def detect_face(bgr_small):
    """-> (found, rotation_degrees, face_area_fraction, kind). Tries 4 rotations because augmented images are often rotated."""
    H, W = bgr_small.shape[:2]
    if FACE_BACKEND == "yunet":
        det = _detector()
        for k in range(4):
            im = np.ascontiguousarray(np.rot90(bgr_small, k))
            det.setInputSize((im.shape[1], im.shape[0]))
            _, fc = det.detect(im)
            if fc is not None and len(fc):
                a = max(float(f[2] * f[3]) for f in fc)
                return True, k * 90, min(1.0, a / (H * W)), "yunet"
        return False, None, 0.0, ""
    front, prof = _detector()
    g = cv2.equalizeHist(cv2.cvtColor(bgr_small, cv2.COLOR_BGR2GRAY))
    for k in range(4):
        gg = np.ascontiguousarray(np.rot90(g, k))
        f = front.detectMultiScale(gg, 1.1, 5, minSize=(int(0.12 * min(gg.shape)),) * 2)
        if len(f):
            return True, k * 90, float(max(w * h for (_, _, w, h) in f)) / (H * W), "haar-frontal"
    for flip in (False, True):
        gg = np.ascontiguousarray(cv2.flip(g, 1) if flip else g)
        f = prof.detectMultiScale(gg, 1.1, 4, minSize=(int(0.15 * min(gg.shape)),) * 2)
        if len(f):
            return True, 0, float(max(w * h for (_, _, w, h) in f)) / (H * W), "haar-profile"
    return False, None, 0.0, ""

def features_for(path):
    try:
        data = np.fromfile(path, np.uint8)
    except Exception as e:
        return {"corrupt": True, "error": "read failed: %s" % e}
    md5 = hashlib.md5(data).hexdigest()
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if img is None:
        return {"corrupt": True, "md5": md5, "error": "cannot be decoded as an image"}
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    thumb = cv2.resize(gray, (32, 32), interpolation=cv2.INTER_AREA)       # used for near-duplicate matching
    g64 = cv2.resize(gray, (64, 64), interpolation=cv2.INTER_AREA)
    corners = [g64[:12, :12], g64[:12, -12:], g64[-12:, :12], g64[-12:, -12:]]
    vignette = bool(all(c.mean() < 25 for c in corners) and g64[20:44, 20:44].mean() > 40)
    face = (False, None, 0.0, "")
    if FACE_BACKEND:
        sc = 320.0 / max(h, w)
        face = detect_face(cv2.resize(img, (max(8, int(w * sc)), max(8, int(h * sc)))) if sc < 1 else img)
    return {"corrupt": False, "md5": md5, "w": w, "h": h, "thumb": thumb, "vignette": vignette,
            "brightness": float(gray.mean()), "sharpness": float(cv2.Laplacian(g64, cv2.CV_64F).var()),
            "face": face[0], "face_rot": face[1], "face_frac": face[2], "face_kind": face[3]}

def feat_job(p):
    st = os.stat(p)
    key = "%s|%d|%d" % (p, st.st_size, st.st_mtime_ns)
    if key in cache and ("thumb" in cache[key] or cache[key].get("corrupt")):      # older caches without thumbnails are recomputed
        return p, cache[key]
    f = features_for(p)
    cache[key] = f
    return p, f

t0 = time.time()
paths = list(df_img.source_path)
res = {}
with ThreadPoolExecutor(max_workers=4) as ex:
    for i, (p, f) in enumerate(ex.map(feat_job, paths)):
        res[p] = f
        if (i + 1) % 2000 == 0:
            print("  %d / %d  (%.0fs)" % (i + 1, len(paths), time.time() - t0))
safe_out(AUDIT_DIR).mkdir(parents=True, exist_ok=True)
pickle.dump(cache, open(safe_out(CACHE), "wb"))
F = [res[p] for p in df_img.source_path]
df_img["corrupt"] = [f.get("corrupt", True) for f in F]
df_img["md5"] = [f.get("md5") for f in F]
df_img["w_px"] = [f.get("w") for f in F]
df_img["h_px"] = [f.get("h") for f in F]
df_img["vignette"] = [f.get("vignette", False) for f in F]
df_img["sharpness"] = [f.get("sharpness") for f in F]
df_img["face_found"] = [f.get("face", False) for f in F]
df_img["face_kind"] = [f.get("face_kind", "") for f in F]
df_img["face_frac"] = [f.get("face_frac", 0.0) for f in F]
df_img["error"] = [f.get("error", "") for f in F]
df_img["width"] = df_img["w_px"].fillna(df_img["width"])
df_img["height"] = df_img["h_px"].fillna(df_img["height"])
print("done in %.0fs | corrupt/unreadable: %d | face found: %d / %d" % (time.time() - t0, df_img.corrupt.sum(), df_img.face_found.sum(), len(df_img)))

## Stage 6 - Annotation & image validation -> issue list
Checks: corrupted images, images without annotations, empty label files, annotations without images, malformed lines, coordinates outside the image,
zero-area boxes/polygons, suspiciously large boxes, unmapped / unconfirmed labels, mixed conditions, low resolution, vignette (dermoscopy-like) images.
**Nothing is deleted** - every finding becomes a row with a *recommended* action.

In [ ]:
issues = []
def add(image_id, source, issue, severity, detail, action):
    issues.append({"image_id": image_id, "source": source, "issue": issue, "severity": severity, "detail": detail, "recommended_action": action})

for _, r in df_img.iterrows():
    for (kind, detail) in r["_probs"]:
        sev = "info" if kind == "no_annotation" else "warn"
        act = {"no_annotation": "exclude from localization training unless you annotate it",
               "empty_label_file": "exclude from localization training unless you annotate it"}.get(kind, "inspect the label file")
        add(r["image_id"], r["source"], kind, sev, detail, act)
    if r["corrupt"]:
        add(r["image_id"], r["source"], "corrupted_image", "error", r["error"], "reject (cannot be used) - confirm in review CSV")
    elif min(r["width"], r["height"]) < LOW_RES_PX:
        add(r["image_id"], r["source"], "low_resolution", "warn", "%dx%d px" % (r["width"], r["height"]), "review; probably too small for localization")
    if r["vignette"]:
        add(r["image_id"], r["source"], "possible_dermoscopy_or_vignette", "warn", "dark circular/corner vignette", "review - may be a dermoscopic / non-facial image")

W = dict(zip(df_img.image_id, zip(df_img.width, df_img.height)))
for a in df_ann.itertuples():
    Wi, Hi = W[a.image_id]
    area = a.w * a.h
    cond = a.standardized_label if a.standardized_label in LARGE_BOX_FRAC else None
    if a.w <= 0 or a.h <= 0 or (Wi and a.w * Wi < 1) or (Hi and a.h * Hi < 1):
        add(a.image_id, a.source, "zero_area_box", "error", "ann %d '%s' %.5f x %.5f" % (a.ann_idx, a.original_label, a.w, a.h), "inspect; likely a labelling error")
    if a.type == "polygon" and len(a.poly) >= 3:
        xs, ys = zip(*a.poly)
        shoelace = 0.5 * abs(sum(xs[i] * ys[(i + 1) % len(xs)] - xs[(i + 1) % len(xs)] * ys[i] for i in range(len(xs))))
        if shoelace <= 0:
            add(a.image_id, a.source, "zero_area_polygon", "error", "ann %d" % a.ann_idx, "inspect; degenerate polygon")
    x0, x1, y0, y1 = a.cx - a.w / 2, a.cx + a.w / 2, a.cy - a.h / 2, a.cy + a.h / 2
    pts = a.poly if a.type == "polygon" else [[x0, y0], [x1, y1]]
    if any(px < -1e-3 or px > 1 + 1e-3 or py < -1e-3 or py > 1 + 1e-3 for px, py in pts):
        add(a.image_id, a.source, "box_outside_image", "warn", "ann %d extends beyond the image" % a.ann_idx, "clip to image or inspect")
    if cond and area > LARGE_BOX_FRAC[cond]:
        add(a.image_id, a.source, "suspicious_large_box", "warn", "ann %d '%s' covers %.0f%% of the image" % (a.ann_idx, a.original_label, area * 100), "inspect; may be a whole-image label")
    if a.standardized_label == "UNMAPPED":
        add(a.image_id, a.source, "label_unmapped", "warn", "'%s' has no mapping" % a.original_label, "decide a mapping in LABEL_MAP_OVERRIDES")
    elif not a.label_confirmed:
        add(a.image_id, a.source, "label_needs_confirmation", "info", "'%s' -> %s is only a proposal" % (a.original_label, a.standardized_label), "confirm once per label in LABEL_MAP_OVERRIDES (Stage 4), not per image")
    if a.standardized_label == "DROP":
        add(a.image_id, a.source, "label_marked_DROP", "info", "'%s' will not be exported" % a.original_label, "none (your decision)")

# mixed conditions inside one image
for iid, g in df_ann.groupby("image_id"):
    labs = {l for l in g.standardized_label if l not in ("DROP", "UNMAPPED")}
    if len(labs) > 1:
        add(iid, g.source.iloc[0], "mixed_conditions", "warn", "labels: %s" % sorted(labs), "review which condition this image belongs to")

for o in orphan_rows:
    add("(none)", o["source"], "annotation_without_image", "warn", o["file"], "check for a missing/renamed image")

df_issues = pd.DataFrame(issues)
print("issues found: %d" % len(df_issues))
if len(df_issues):
    display(df_issues.groupby(["issue", "severity"]).size().rename("count").reset_index().sort_values("count", ascending=False))

## Stage 7 - Duplicate and near-duplicate groups
Three signals are combined: **exact** (identical file), **name** (same Roboflow base name, e.g. `acne-269_jpeg`, in any source), and **near** (32x32 thumbnail
correlation >= `NEAR_DUP_MIN_CORR`, compared against all 8 rotations/flips, so grayscale / noisy / rotated / recoloured copies of one photo are caught).
These thresholds were **calibrated on your data**: a plain 64-bit perceptual hash both missed real copies and chained unrelated photos into one 6,000-image group.
`MAX_GROUP_SIZE` skips any link that would create a huge group and reports it. Every image gets a `duplicate_group` id - use it later so a group never
crosses train/validation/test. Review the largest groups visually in Stage 9.

In [ ]:
n = len(df_img)
parent = list(range(n))
gsize = [1] * n
skipped = collections.Counter()
reason = collections.defaultdict(set)

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x

def union(a, b, why):
    ra, rb = find(a), find(b)
    if ra == rb:
        reason[a].add(why); reason[b].add(why); return
    if gsize[ra] + gsize[rb] > MAX_GROUP_SIZE:
        skipped[why] += 1; return
    parent[rb] = ra; gsize[ra] += gsize[rb]
    reason[a].add(why); reason[b].add(why)

ok = [i for i in range(n) if not df_img.corrupt.iloc[i]]

# 1) exact
by_md5 = collections.defaultdict(list)
for i in ok:
    by_md5[df_img.md5.iloc[i]].append(i)
for g in by_md5.values():
    for j in g[1:]:
        union(g[0], j, "exact")

# 2) same Roboflow base name (any source)
def name_key(fn):
    s_ = Path(fn).name.split(".rf.")[0]
    s_ = re.sub(r"\.(jpe?g|png|webp|bmp)$", "", s_, flags=re.I)
    return re.sub(r"(_(jpe?g|png|webp))+$", "", s_, flags=re.I).lower()
if USE_NAME_GROUPS:
    by_name = collections.defaultdict(list)
    for i in range(n):
        k = name_key(df_img.original_filename.iloc[i])
        if len(k) >= 6:
            by_name[k].append(i)
    for g in by_name.values():
        for j in g[1:]:
            union(g[0], j, "name")

# 3) near-duplicate: thumbnail correlation, best of 8 rotations/flips
def zn(x):
    x = x.reshape(x.shape[0], -1).astype(np.float32)
    x = x - x.mean(1, keepdims=True)
    return (x / (x.std(1, keepdims=True) + 1e-6) / np.sqrt(x.shape[1])).astype(np.float32)
TH = np.stack([res[df_img.source_path.iloc[i]]["thumb"] for i in ok]).astype(np.float32)       # (m, 32, 32)
m_ok = len(ok)
base = zn(TH)
variants = []
for k in range(4):
    r_ = np.rot90(TH, k, axes=(-2, -1))
    variants += [r_, r_[..., ::-1]]
edges = []                                            # memory-light: one orientation (~70 MB) at a time, small chunks
for v_ in variants:
    Vk = zn(np.ascontiguousarray(v_))
    for s0 in range(0, m_ok, 512):
        c = base[s0:s0 + 512] @ Vk.T
        rr, cc = np.nonzero(c >= NEAR_DUP_MIN_CORR)
        for r_i, c_i in zip(rr, cc):
            a_i, b_i = s0 + int(r_i), int(c_i)
            if a_i < b_i:
                edges.append((float(c[r_i, c_i]), a_i, b_i))
    del Vk
edges.sort(reverse=True)                              # strongest links first
for _, a_, b_ in edges:
    union(ok[a_], ok[b_], "near")

roots, gid = {}, []
for i in range(n):
    r_ = find(i)
    roots.setdefault(r_, "DG-%06d" % (len(roots) + 1))
    gid.append(roots[r_])
df_img["duplicate_group"] = gid
df_img["duplicate_signals"] = [",".join(sorted(reason[i])) for i in range(n)]
gs = df_img.groupby("duplicate_group").agg(size=("image_id", "size"), sources=("source", "nunique"))
print("images: %d | duplicate groups: %d | groups with >1 image: %d | images inside multi-image groups: %d" % (
    n, len(gs), (gs["size"] > 1).sum(), gs[gs["size"] > 1]["size"].sum()))
print("near-duplicate links: %d | groups spanning several sources: %d | largest group: %d images | links skipped by MAX_GROUP_SIZE: %s" % (
    len(edges), (gs.sources > 1).sum(), gs["size"].max(), dict(skipped) or "none"))
display(gs["size"].value_counts().sort_index().rename("groups").to_frame().T)
for r in df_img[df_img.duplicate_signals != ""].itertuples():
    add(r.image_id, r.source, "duplicate_group_member", "info", "group %s (%d images, signals: %s)" % (r.duplicate_group, gs.loc[r.duplicate_group, "size"], r.duplicate_signals),
        "keep, but never split this group across train/val/test")
df_issues = pd.DataFrame(issues)

## Stage 8 - Anatomical site: facial / non_facial / needs_manual_review
Rules (conservative - **no guessing**):
1. If you set the site yourself in `SITE_OVERRIDES` -> that wins.
2. If you declared the whole source in `SOURCE_SITE_DECLARATION` (only after browsing it), or the source folder name declares it (`FACIAL_DECLARED_NAME_WORDS`, e.g. *facial*-eczema) -> `facial` (basis: source name).
3. If a face is detected -> `facial` (basis: detector).
4. Otherwise -> `needs_manual_review`. **Nothing is ever auto-labelled `non_facial`** - only you can do that.

Eczema and Hyperpigmentation images that are not confirmed facial are flagged for review (your system analyses faces); for Acne it is informational only.

In [ ]:
site, basis = [], []
declared = {s: any(w in s.lower() for w in FACIAL_DECLARED_NAME_WORDS) for s in df_img.source.unique()}
for r in df_img.itertuples():
    if r.image_id in SITE_OVERRIDES:
        site.append(SITE_OVERRIDES[r.image_id]); basis.append("manual override")
    elif r.source in SOURCE_SITE_DECLARATION:
        site.append(SOURCE_SITE_DECLARATION[r.source]); basis.append("declared by you for this source")
    elif declared[r.source]:
        site.append("facial"); basis.append("source name declares facial")
    elif r.face_found:
        site.append("facial"); basis.append("face detector (%s, %.0f%% of frame)" % (r.face_kind, 100 * r.face_frac))
    else:
        site.append("needs_manual_review"); basis.append("no face found - cannot decide automatically")
df_img["facial_status"] = site
df_img["facial_basis"] = basis
df_img["anatomical_site"] = ["face" if s == "facial" else ("body/other (manual)" if s == "non_facial" else "unknown") for s in site]

# image-level condition from the confirmed/proposed standardized labels
cond_map = {}
for iid, g in df_ann.groupby("image_id"):
    labs = sorted({l for l in g.standardized_label if l not in ("DROP", "UNMAPPED")})
    cond_map[iid] = labs[0] if len(labs) == 1 else ("Mixed" if len(labs) > 1 else None)
df_img["condition"] = [cond_map.get(i) or c for i, c in zip(df_img.image_id, df_img.source_condition)]

for r in df_img.itertuples():
    if r.facial_status == "needs_manual_review":
        sev = "warn" if r.condition in ("Eczema", "Hyperpigmentation") else "info"
        add(r.image_id, r.source, "anatomical_site_unknown", sev, r.facial_basis, "browse it (Stage 9); set SITE_OVERRIDES to facial / non_facial")
    if r.condition == "Eczema" and r.facial_status == "non_facial":
        add(r.image_id, r.source, "non_facial_eczema", "info", "kept separate; never place in a facial val/test set", "supplementary training only")
cm = df_img.groupby("duplicate_group").condition.nunique()
mixed_groups = set(cm[cm > 1].index)
for r in df_img[df_img.duplicate_group.isin(mixed_groups)].itertuples():
    add(r.image_id, r.source, "duplicate_across_conditions", "warn", "group %s contains the same photo under several conditions" % r.duplicate_group,
        "browse the group (Stage 9, duplicate_group filter): the photo may carry the wrong condition")
print("duplicate groups containing more than one condition (same photo in e.g. an acne AND a pigmentation dataset): %d" % len(mixed_groups))
df_issues = pd.DataFrame(issues)
display(pd.crosstab([df_img.condition.fillna("?"), df_img.source], df_img.facial_status))

## Stage 9 - Visual browser  (look at every image)
Shows images with their **boxes (red) / polygons (cyan)**. Set the variables and re-run the cell. Increase `START_INDEX` by `BATCH_SIZE` to move to the next batch.
`BROWSE_FILTER` narrows the list (any field can be `None`). Each tile shows: image id, source, original filename, condition, original label(s), annotation type,
site, duplicate group and any issues.

In [ ]:
START_INDEX = 0
BATCH_SIZE = 12
BROWSE_FILTER = {
    "condition": None,          # "Acne" | "Eczema" | "Hyperpigmentation"
    "source": None,             # part of a source folder name, e.g. "Eczema.v1"
    "facial_status": None,      # "facial" | "needs_manual_review" | "non_facial"
    "issue": None,              # e.g. "suspicious_large_box", "no_annotation", "label_needs_confirmation"
    "duplicate_group": None,    # e.g. "DG-000012" - see all copies of one photo
    "image_id": None,
    "label": None,              # (source substring, original label) e.g. ("acne-detection.v2i", "spot") - that label's boxes are drawn RED, others GREY
}

def issues_of(image_id):
    if not len(df_issues):
        return ""
    return ",".join(sorted(set(df_issues.loc[df_issues.image_id == image_id, "issue"])))

def select_for_browser():
    d = df_img
    f = BROWSE_FILTER
    if f["condition"]: d = d[d.condition == f["condition"]]
    if f["source"]: d = d[d.source.str.contains(f["source"], case=False, regex=False)]
    if f["facial_status"]: d = d[d.facial_status == f["facial_status"]]
    if f["duplicate_group"]: d = d[d.duplicate_group == f["duplicate_group"]]
    if f["image_id"]: d = d[d.image_id == f["image_id"]]
    if f["label"]:
        sub, lab = f["label"]
        ids = set(df_ann[df_ann.source.str.contains(sub, case=False, regex=False) & (df_ann.original_label.astype(str) == str(lab))].image_id)
        d = d[d.image_id.isin(ids)]
    if f["issue"]: d = d[d.image_id.isin(set(df_issues.loc[df_issues.issue == f["issue"], "image_id"]))]
    return d.reset_index(drop=True)

def draw(img, anns):
    H, W = img.shape[:2]
    out = img.copy()
    th = max(2, int(max(H, W) / 250))
    focus = str(BROWSE_FILTER["label"][1]) if BROWSE_FILTER.get("label") else None
    for a in anns:
        if focus is not None and str(a["original_label"]) != focus:
            red, cyan = (140, 140, 140), (140, 140, 140)              # other labels shown in grey
        else:
            red, cyan = (0, 0, 255), (255, 255, 0)
        if a["type"] == "polygon" and a["poly"]:
            pts = np.array([[int(x * W), int(y * H)] for x, y in a["poly"]], np.int32)
            cv2.polylines(out, [pts], True, cyan, th)                # BGR cyan
        else:
            p0 = (int((a["cx"] - a["w"] / 2) * W), int((a["cy"] - a["h"] / 2) * H))
            p1 = (int((a["cx"] + a["w"] / 2) * W), int((a["cy"] + a["h"] / 2) * H))
            cv2.rectangle(out, p0, p1, red, th)
    return out

def browse(start=None, size=None):
    start = START_INDEX if start is None else start
    size = BATCH_SIZE if size is None else size
    sel = select_for_browser()
    batch = sel.iloc[start:start + size]
    print("showing %d-%d of %d matching images   (legend: red = box, cyan = polygon/mask outline)" % (start, start + len(batch) - 1, len(sel)))
    if not len(batch):
        return
    cols = min(BROWSER_COLUMNS, len(batch))
    rows_n = int(math.ceil(len(batch) / cols))
    fig, axs = plt.subplots(rows_n, cols, figsize=(4.6 * cols, 5.4 * rows_n), squeeze=False)
    for ax in axs.ravel():
        ax.axis("off")
    for ax, r in zip(axs.ravel(), batch.itertuples()):
        img = imread_any(r.source_path)
        anns = [a for a in df_img.loc[df_img.image_id == r.image_id, "_anns"].iloc[0]]
        if img is None:
            ax.text(0.5, 0.5, "UNREADABLE", ha="center"); continue
        ax.imshow(cv2.cvtColor(draw(img, anns), cv2.COLOR_BGR2RGB))
        labs = sorted({a["original_label"] for a in anns}) or ["-"]
        ax.set_title("%s | %s\n%s\n%s | %s | %d ann (%s)\nsite: %s | %s\n%s" % (
            r.image_id, r.condition, r.original_filename[:40], r.source[:34], ",".join(labs), len(anns),
            r.annotation_format, r.facial_status, r.duplicate_group, issues_of(r.image_id)[:60]), fontsize=7, loc="left")
    plt.tight_layout(); plt.show()

browse()

In [ ]:
# Quick helpers - run any of these, then re-run the browse() cell above.
# 1) The 15 largest duplicate groups (see whether they really are the same photo):
big = df_img.groupby("duplicate_group").size().sort_values(ascending=False).head(15)
display(big.rename("images").to_frame())
# Example: BROWSE_FILTER["duplicate_group"] = big.index[0]; START_INDEX = 0; browse()

## Stage 10 - Review CSV  (you decide: keep / reject / review)
Writes `01_audit_reports/review_decisions.csv` with one row per image that has a warning/error, plus a recommended action. Open it in Excel/VS Code, fill the
`user_decision` column with **keep**, **reject** or **review**, save, and re-run this cell. Your existing decisions are preserved when the file is regenerated.
Defaults (until you decide): clean images = keep; images with an error/warning = review. **Nothing is ever rejected automatically.**

In [ ]:
REVIEW_CSV = AUDIT_DIR / "review_decisions.csv"
sev_rank = {"error": 3, "warn": 2, "info": 1}
need = df_issues[df_issues.severity.isin(["warn", "error"]) & (df_issues.image_id != "(none)")]
agg = need.groupby("image_id").agg(issues=("issue", lambda s: ", ".join(sorted(set(s)))),
                                   recommended_action=("recommended_action", lambda s: " | ".join(sorted(set(s)))[:300]),
                                   worst=("severity", lambda s: max(s, key=lambda x: sev_rank[x]))).reset_index()
rev = agg.merge(df_img[["image_id", "source", "original_filename", "condition", "facial_status", "duplicate_group"]], on="image_id", how="left")
rev["user_decision"] = ""
if REVIEW_CSV.exists():
    old = pd.read_csv(REVIEW_CSV, dtype=str).fillna("")
    rev = rev.merge(old[["image_id", "user_decision"]].rename(columns={"user_decision": "_old"}), on="image_id", how="left")
    rev["user_decision"] = rev["_old"].fillna("")
    rev = rev.drop(columns="_old")
rev = rev[["image_id", "original_filename", "source", "condition", "facial_status", "duplicate_group", "issues", "worst", "recommended_action", "user_decision"]]
safe_out(AUDIT_DIR).mkdir(parents=True, exist_ok=True)
rev.to_csv(safe_out(REVIEW_CSV), index=False)
df_issues.to_csv(safe_out(AUDIT_DIR / "all_issues.csv"), index=False)
if len(orphan_rows):
    pd.DataFrame(orphan_rows).to_csv(safe_out(AUDIT_DIR / "orphan_annotation_files.csv"), index=False)

decision = dict(zip(rev.image_id, rev.user_decision.str.lower()))
bad = {k: v for k, v in decision.items() if v not in ("", "keep", "reject", "review")}
assert not bad, "user_decision must be keep / reject / review: %s" % list(bad.items())[:5]
final = []
flagged = set(rev.image_id)
for iid in df_img.image_id:
    d = decision.get(iid, "")
    final.append(d if d else ("review" if iid in flagged else "keep"))
df_img["decision"] = final
df_img["review_status"] = ["reviewed by you" if decision.get(i) else ("flagged" if i in flagged else "clean") for i in df_img.image_id]
print("review file:", REVIEW_CSV, "| rows:", len(rev))
print(df_img.decision.value_counts().to_string())
print("\nflagged by severity:"); print(rev.worst.value_counts().to_string())

## Stage 11 - Proposed final structure (nothing is created yet)
Raw, review and final data stay separate. Boxes, polygons and masks coexist because the **source of truth is one JSON per image** (all
annotations with original + standardized labels and mapping history); YOLO/COCO exports are generated later from these when you split.

In [ ]:
print("""
PROPOSED LAYOUT   (OUT_ROOT = %s)

00_raw_extracted/                 only archives you allowed to be extracted (originals are never touched)
01_audit_reports/                 label_mapping_proposal.csv, all_issues.csv, review_decisions.csv, features_cache.pkl,
                                  master_metadata.csv, annotations_long.csv, duplicate_groups.csv, audit_report.md/json
02_review_queue/images/<Condition>/  + annotations/<Condition>/   images with decision = review (copied, for you to inspect)
03_standardized/
    images/<Condition>/<facial|non_facial|needs_manual_review>/IMG-xxxxxx.<ext>     decision = keep
    annotations/<Condition>/IMG-xxxxxx.json                                          boxes + polygons, original+standard labels
    metadata/master_metadata.csv, annotations_long.csv, duplicate_groups.csv

Not created here:  train/ valid/ test/   (split later by duplicate_group, facial-only val/test for facial claims)
""" % OUT_ROOT)

## Stage 12 - Reorganise (dry-run first, then apply)
Copies **only** (never moves/deletes) `keep` images into `03_standardized` and `review` images into `02_review_queue`; `reject` images are listed, not copied.
While `APPLY_CHANGES = False` this prints the plan. To execute: set `APPLY_CHANGES = True` in Stage 0, run Stage 0 again, then run this cell.

In [ ]:
INCLUDE_REVIEW_IN_STANDARDIZED = False      # True = also copy 'review' images into 03_standardized (not recommended before you review)

def dst_for(r):
    cond = r.condition if r.condition in STANDARD_CONDITIONS else "Unassigned"
    ext = Path(r.original_filename).suffix.lower() or ".jpg"
    if r.decision == "keep" or (r.decision == "review" and INCLUDE_REVIEW_IN_STANDARDIZED):
        return FINAL_DIR / "images" / cond / r.facial_status / (r.image_id + ext)
    if r.decision == "review":
        return REVIEW_DIR / "images" / cond / (r.image_id + ext)
    return None

df_img["current_filename"] = ""
plan = []
for r in df_img.itertuples():
    d = dst_for(r)
    if d is not None:
        plan.append((r.image_id, r.source_path, d))
        df_img.loc[df_img.image_id == r.image_id, "current_filename"] = str(d.relative_to(OUT_ROOT))
print("PLAN: %d image copies | keep %d | review %d | reject (not copied) %d" % (
    len(plan), (df_img.decision == "keep").sum(), (df_img.decision == "review").sum(), (df_img.decision == "reject").sum()))
print("per condition:"); print(df_img.groupby(["condition", "decision"], dropna=False).size().unstack(fill_value=0).to_string())
for p in plan[:8]:
    print("  ", p[1], "->", p[2].relative_to(OUT_ROOT))

def annotation_json(r):
    anns = df_ann[df_ann.image_id == r.image_id]
    return {"image_id": r.image_id, "source_dataset": r.source, "source_path": r.source_path, "original_filename": r.original_filename,
            "source_split": r.source_split, "width": int(r.width) if pd.notna(r.width) else None, "height": int(r.height) if pd.notna(r.height) else None,
            "condition": r.condition, "annotation_format": r.annotation_format, "anatomical_site": r.anatomical_site,
            "facial_status": r.facial_status, "facial_basis": r.facial_basis, "duplicate_group": r.duplicate_group,
            "review_status": r.review_status, "decision": r.decision,
            "annotations": [{"original_label": a.original_label, "standardized_label": a.standardized_label, "label_confirmed": bool(a.label_confirmed),
                             "mapping_history": a.mapping_history, "type": a.type,
                             "bbox_xywh_norm": [a.cx - a.w / 2, a.cy - a.h / 2, a.w, a.h],
                             "polygon_norm": a.poly} for a in anns.itertuples() if a.standardized_label != "DROP"]}

if APPLY_CHANGES and (df_labels.confirmed != "yes").any() and not ALLOW_UNCONFIRMED_LABELS:
    raise SystemExit("STOP: %d label mapping(s) are still unconfirmed (see Stage 4). Put your decisions in LABEL_MAP_OVERRIDES, "
                     "re-run Stage 4 onwards, then apply. (ALLOW_UNCONFIRMED_LABELS=True overrides - not recommended.)" % (df_labels.confirmed != "yes").sum())

if not APPLY_CHANGES:
    print("\nDRY-RUN ONLY. Nothing copied. To apply: set APPLY_CHANGES = True (Stage 0), re-run Stage 0, then this cell.")
else:
    copied = failed = 0
    for iid, src, dst in plan:
        safe_out(dst).parent.mkdir(parents=True, exist_ok=True)
        try:
            shutil.copy2(src, dst)
            if os.path.getsize(src) != os.path.getsize(dst):
                raise IOError("size mismatch after copy")
            copied += 1
        except Exception as e:
            failed += 1; print("FAILED", iid, e)
    for r in df_img.itertuples():
        if r.decision in ("keep", "review") and dst_for(r) is not None:
            cond = r.condition if r.condition in STANDARD_CONDITIONS else "Unassigned"
            base = FINAL_DIR if (r.decision == "keep" or INCLUDE_REVIEW_IN_STANDARDIZED) else REVIEW_DIR
            ap = safe_out(base / "annotations" / cond / (r.image_id + ".json"))
            ap.parent.mkdir(parents=True, exist_ok=True)
            ap.write_text(json.dumps(annotation_json(r), indent=1))
    print("\nAPPLIED: copied %d | failed %d | originals untouched (read-only)" % (copied, failed))
    assert copied == len(plan) - failed

In [ ]:
# Master metadata + long annotation table + duplicate groups  (always written to 01_audit_reports; copied beside the data if applied)
cols = ["image_id", "original_filename", "current_filename", "condition", "source_condition", "source", "source_path", "source_split", "annotation_format",
        "annotation_path", "anatomical_site", "facial_status", "facial_basis", "duplicate_group", "duplicate_signals", "width", "height",
        "corrupt", "review_status", "decision"]
agg_lab = df_ann.groupby("image_id").agg(original_label=("original_label", lambda s: ";".join(sorted(set(s)))),
                                         standardized_label=("standardized_label", lambda s: ";".join(sorted(set(s)))),
                                         n_boxes=("type", lambda s: int((s == "bbox").sum())),
                                         n_polygons=("type", lambda s: int((s == "polygon").sum())),
                                         mapping_history=("mapping_history", lambda s: " | ".join(sorted(set(s)))))
master = df_img[cols].merge(agg_lab, on="image_id", how="left")
master["annotation_type"] = master.annotation_format
master["quality_review_status"] = master.review_status
master = master.rename(columns={"source": "source_dataset"})
safe_out(AUDIT_DIR).mkdir(parents=True, exist_ok=True)
master.to_csv(safe_out(AUDIT_DIR / "master_metadata.csv"), index=False)
df_ann.drop(columns=["poly"]).to_csv(safe_out(AUDIT_DIR / "annotations_long.csv"), index=False)
df_img.groupby("duplicate_group").agg(images=("image_id", "size"), sources=("source", lambda s: ";".join(sorted(set(s)))),
                                      image_ids=("image_id", lambda s: ";".join(s))).to_csv(safe_out(AUDIT_DIR / "duplicate_groups.csv"))
if APPLY_CHANGES:
    meta = safe_out(FINAL_DIR / "metadata"); meta.mkdir(parents=True, exist_ok=True)
    for f in ("master_metadata.csv", "annotations_long.csv", "duplicate_groups.csv"):
        shutil.copy2(AUDIT_DIR / f, meta / f)
print("written to", AUDIT_DIR)
display(master.head(10))

## Stage 13 - Final audit report
Totals per condition, facial vs non-facial, images per source, box/polygon counts, corrupted files, missing annotations, duplicate groups, rejected and
to-review images, plus the folder tree. Saved as `audit_report.md` / `audit_report.json`.

In [ ]:
def tree(root, max_depth=6, prefix=""):
    root = Path(root)
    if not root.exists():
        return prefix + "(does not exist yet - nothing applied)\n"
    out, kids = "", sorted(root.iterdir())
    dirs = [k for k in kids if k.is_dir()]
    nfiles = sum(1 for k in kids if k.is_file())
    if nfiles:
        out += "%s%d file(s)\n" % (prefix, nfiles)
    for d in dirs:
        out += "%s%s/\n" % (prefix, d.name)
        if len(prefix) // 2 < max_depth:
            out += tree(d, max_depth, prefix + "  ")
    return out

summary = {
    "total_images_audited": int(len(df_img)),
    "images_by_condition": df_img.condition.fillna("Unassigned").value_counts().to_dict(),
    "images_by_facial_status": df_img.facial_status.value_counts().to_dict(),
    "images_by_source": df_img.source.value_counts().to_dict(),
    "bounding_boxes": int((df_ann.type == "bbox").sum()),
    "polygons_masks": int((df_ann.type == "polygon").sum()),
    "annotations_by_standardized_label": df_ann.standardized_label.value_counts().to_dict(),
    "corrupted_files": int(df_img.corrupt.sum()),
    "images_missing_annotations": int(df_issues[df_issues.issue.isin(["no_annotation", "empty_label_file"])].image_id.nunique()),
    "annotation_files_without_image": len(orphan_rows),
    "duplicate_groups_total": int(df_img.duplicate_group.nunique()),
    "duplicate_groups_with_multiple_images": int((df_img.groupby("duplicate_group").size() > 1).sum()),
    "images_in_duplicate_groups": int(df_img.duplicate_group.map(df_img.duplicate_group.value_counts()).gt(1).sum()),
    "images_with_near_duplicate_signal": int(df_img.duplicate_signals.str.contains("near").sum()),
    "decision_counts": df_img.decision.value_counts().to_dict(),
    "images_requiring_manual_review": int((df_img.decision == "review").sum()),
    "labels_awaiting_confirmation": int((df_labels.confirmed != "yes").sum()),
    "facial_eczema": int(((df_img.condition == "Eczema") & (df_img.facial_status == "facial")).sum()),
    "eczema_needing_site_review": int(((df_img.condition == "Eczema") & (df_img.facial_status == "needs_manual_review")).sum()),
    "apply_changes": bool(APPLY_CHANGES),
}
for k, v in summary.items():
    print("%-42s %s" % (k, v))
md_lines = ["# PrecisionSkin localization dataset audit", ""] + ["- **%s**: %s" % (k, v) for k, v in summary.items()]
safe_out(AUDIT_DIR).mkdir(parents=True, exist_ok=True)
safe_out(AUDIT_DIR / "audit_report.md").write_text("\n".join(md_lines))
safe_out(AUDIT_DIR / "audit_report.json").write_text(json.dumps(summary, indent=2, default=str))
print("\nFOLDER TREE of", OUT_ROOT)
print(OUT_ROOT.name + "/"); print(tree(OUT_ROOT))
print("Next (not done here): choose the keep/reject decisions, confirm labels and sites, then split by duplicate_group.")